# Neuronal Tracer — Step 2 v3: GWDT-based Medial Axis Extraction

**Input:** `output/tubularity.npz`, `output/stack_iso.tif` (from Step 1)

**Output:** `output/skeleton.npz`

**Next:** `step3.ipynb` — soma detection, competitive FMM, SWC export

## Core idea — APP2 Gray-Weighted Distance Transform

No binarisation. The soft tubularity map is used directly as a speed field.

```
cost(v)  = 1 / (T(v) + ε)     ← bright voxels are cheap to traverse
GWDT(v)  = minimum weighted path length from v to background (T ≈ 0)
         = how "deep inside bright structure" voxel v is
ridge    = local maximum of GWDT  →  tube centerline
```

The FMM front is seeded where `T ≈ 0` (true background — no explicit threshold).
Bright fibers create corridors of low cost; their centers accumulate the highest GWDT value.

At a **crossing**, the two fibers have different T profiles in cross-section →
their GWDT ridges diverge → separate centerlines emerge naturally.

| Step | Description |
|------|-------------|
| 5a | **GWDT** — FMM from T≈0 seeds, speed ∝ T^α |
| 5b | **Ridge extraction** — local maxima of GWDT = medial axis |
| 5c | **Size filter** — remove tiny ridge components |
| 5d | **Thinning** — Lee 1994 → 1-voxel centerlines |

In [ ]:
# Run once, then restart kernel.
# scikit-fmm provides the Fast Marching Method solver for GWDT computation.
# !pip install tifffile scikit-image scipy numpy matplotlib ipywidgets scikit-fmm

In [ ]:
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.ndimage import maximum_filter
from skimage import morphology
import gc
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 100,
    'figure.facecolor': '#111111',
    'axes.facecolor': '#111111',
    'axes.titlecolor': 'white',
    'text.color': 'white',
})

# scikit-fmm provides the FMM solver for GWDT.
try:
    import skfmm
    SKFMM_AVAILABLE = True
    print(f'scikit-fmm available — GWDT via FMM enabled.')
except ImportError:
    SKFMM_AVAILABLE = False
    print('scikit-fmm not installed.')
    print('  Install with: pip install scikit-fmm')
    print('  Falling back to binary + Lee skeletonisation (Step 1 behaviour).')

print('All packages loaded.')

---
## Configuration

### GWDT parameters

- **`SPEED_ALPHA`** — exponent for speed function: `speed = (T + SPEED_EPS)^SPEED_ALPHA`.
  α=1: speed ∝ T (default).  α>1: sharpens tube/background contrast.
  Higher α → narrower ridges, better crossing separation.

- **`SPEED_EPS`** — floor added to T before raising to α.
  Prevents zero speed in background.  Larger value → background traversal
  is less penalised (ridges become less sharp).  Typical: 0.005–0.02.

- **`MIN_RIDGE_VOXELS`** — minimum ridge component size (voxels).

There is **no binarisation threshold**.
Seeds are defined implicitly as voxels where `T_combined ≤ SEED_T_MAX` (near-zero
tubularity — genuine background). All other voxels participate in GWDT propagation.

- **`SEED_T_MAX`** — maximum T value still treated as a background seed.
  Default 0.0 uses only exact-zero T voxels.  Raise slightly (e.g. 0.005) if the
  FMM reports "no interface found" (meaning T has no exact zeros).

In [ ]:
# ── Input Paths ───────────────────────────────────────────────────────────────
TUBULARITY_CHECKPOINT = 'output/tubularity.npz'
STACK_ISO_PATH        = 'output/stack_iso.tif'

# ── GWDT Parameters ────────────────────────────────────────────────────────────
# Speed function exponent: speed = (T + SPEED_EPS)^SPEED_ALPHA
# No binarisation threshold — seeds are voxels with T ≤ SEED_T_MAX.
SPEED_ALPHA    = 1.0    # 1.0 = linear; >1 sharpens tube contrast
SPEED_EPS      = 0.005  # floor to prevent zero speed in background
SEED_T_MAX     = 0.0    # T threshold for background seeds (0 = exact zeros only)
                         # Raise to 0.005 if FMM reports no interface found.

# Minimum connected component size for ridge map (voxels)
MIN_RIDGE_VOXELS = 50

print('Configuration loaded.')
print(f'  Tubularity checkpoint : {TUBULARITY_CHECKPOINT}')
print(f'  SPEED_ALPHA           : {SPEED_ALPHA}')
print(f'  SPEED_EPS             : {SPEED_EPS}')
print(f'  SEED_T_MAX            : {SEED_T_MAX}  (background seed threshold)')
print(f'  MIN_RIDGE_VOXELS      : {MIN_RIDGE_VOXELS}')
print()
print('No binarisation threshold. GWDT computed directly on soft tubularity map.')

In [ ]:
# ── Load Step 1 Checkpoint ────────────────────────────────────────────────────
ck = np.load(TUBULARITY_CHECKPOINT)
T_combined   = ck['T_combined']          # float32 (Z, Y, X)
radius_map   = ck['radius_map']          # float32 (Z, Y, X)
voxel_iso    = float(ck['voxel_iso'])    # scalar µm

# orient_field is optional — present in v3+ checkpoints
if 'orient_field' in ck:
    orient_field = ck['orient_field']    # float16 (Z, Y, X, 3)
    print(f'  orient_field loaded : {orient_field.shape}')
else:
    orient_field = None
    print('  orient_field not found in checkpoint (step1 v2 or earlier).')

stack_iso = tifffile.imread(STACK_ISO_PATH)

print(f'T_combined   : {T_combined.shape}  [{T_combined.min():.4f}, {T_combined.max():.4f}]')
print(f'radius_map   : [{radius_map.min():.4f}, {radius_map.max():.4f}] µm')
print(f'voxel_iso    : {voxel_iso:.4f} µm')
print(f'stack_iso    : {stack_iso.shape}  {stack_iso.dtype}')
print(f'RAM (T+stack+radius): {(T_combined.nbytes+stack_iso.nbytes+radius_map.nbytes)/1e9:.2f} GB')

---
## Visualisation Utilities

In [ ]:
def show_mip(vol, title='MIP', cmap='gray', vmin=None, vmax=None):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
    kw = dict(cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
    axes[0].imshow(vol.max(axis=0), aspect='equal', **kw); axes[0].set_title('XY')
    axes[1].imshow(vol.max(axis=1), aspect='equal', **kw); axes[1].set_title('XZ')
    axes[2].imshow(vol.max(axis=2), aspect='equal', **kw); axes[2].set_title('YZ')
    for ax in axes: ax.axis('off')
    fig.suptitle(title, fontsize=13)
    plt.show()


def show_slices(vol, title='', cmap='gray', vmin=None, vmax=None,
                overlay=None, overlay_cmap='hot', overlay_thresh=0.1):
    nz  = vol.shape[0]
    out = widgets.Output()
    slider = widgets.IntSlider(value=nz//2, min=0, max=nz-1,
                               description='Z', layout=widgets.Layout(width='500px'))
    tslider = (widgets.FloatSlider(value=overlay_thresh, min=0.01, max=0.99, step=0.01,
                                    description='Overlay>=',
                                    layout=widgets.Layout(width='350px'))
               if overlay is not None else None)

    def _draw(z, thresh=overlay_thresh):
        with out:
            clear_output(wait=True)
            ncols = 4 if overlay is not None else 3
            fig, axes = plt.subplots(1, ncols, figsize=(5*ncols, 5), constrained_layout=True)
            kw = dict(cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
            axes[0].imshow(vol[z], aspect='equal', **kw); axes[0].set_title(f'XY  z={z}')
            axes[1].imshow(vol[:, vol.shape[1]//2, :], aspect='equal', **kw); axes[1].set_title('XZ centre')
            axes[2].imshow(vol[:, :, vol.shape[2]//2], aspect='equal', **kw); axes[2].set_title('YZ centre')
            if overlay is not None:
                ov = overlay[z].astype(np.float32)
                thr_abs = thresh * max(float(overlay.max()), 1e-10)
                axes[3].imshow(vol[z], cmap='gray', aspect='equal',
                               vmin=vmin, vmax=vmax, interpolation='nearest')
                axes[3].imshow(np.where(ov >= thr_abs, ov, np.nan),
                               cmap=overlay_cmap, alpha=0.6, aspect='equal',
                               interpolation='nearest')
                axes[3].set_title(f'Overlay  thr={thresh:.2f}')
            for ax in axes: ax.axis('off')
            plt.show()

    controls = [slider] + ([tslider] if tslider else [])
    widgets.interact(_draw, z=slider,
                     **({'thresh': tslider} if tslider else {}))
    display(out)


def compare_mip(va, vb, title_a='A', title_b='B', cmap='gray'):
    fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
    for row, (vol, ttl) in enumerate([(va, title_a), (vb, title_b)]):
        kw = dict(cmap=cmap, interpolation='nearest')
        axes[row, 0].imshow(vol.max(axis=0), aspect='equal', **kw); axes[row, 0].set_title(f'{ttl} — XY')
        axes[row, 1].imshow(vol.max(axis=1), aspect='equal', **kw); axes[row, 1].set_title(f'{ttl} — XZ')
        axes[row, 2].imshow(vol.max(axis=2), aspect='equal', **kw); axes[row, 2].set_title(f'{ttl} — YZ')
        for ax in axes[row]: ax.axis('off')
    plt.show()

print('Visualisation utilities defined.')

---
## Step 5a · GWDT via Fast Marching Method (no binarisation)

**APP2 core principle** (Xiao et al. 2013):

```
phi(v)  = T(v) − SEED_T_MAX          signed: + = foreground, − = background seed
speed(v) = (T(v) + SPEED_EPS)^SPEED_ALPHA    bright → fast propagation
GWDT(v)  = FMM travel time from phi < 0 surface to v
```

With `SEED_T_MAX ≈ 0`, seeds are only genuine background voxels (T = 0).
No binary threshold: every non-zero-T voxel contributes to the GWDT.

**Why ridges = centerlines:**
A tube center voxel has both (a) high T and (b) a long path to background.
The GWDT compounds these: the fastest path from tube center to background must travel
through the full cross-section of the tube. Centers accumulate the highest travel time
→ they are local maxima of GWDT → they ARE the ridges.

**Crossing fibers:** two crossing tubes have orthogonal T profiles →
their GWDT landscapes have separate ridges even at the crossing point.

**Memory note:** scikit-fmm uses float64 internally.
For a 1000×1000×500 volume: ~4 GB. Use DOWNSAMPLE_XY=2 in Step 1 if RAM is tight.

In [ ]:
import time as _time

if not SKFMM_AVAILABLE:
    print('scikit-fmm not available — binary + Lee skeletonisation fallback.')
    from skimage.morphology import skeletonize
    from scipy.ndimage import label as _ndlabel
    # Conservative binary mask: use T > 5% of max as proxy for foreground
    _thr  = 0.05 * float(T_combined.max())
    ridges = T_combined > _thr
    lbl, _ = _ndlabel(ridges)
    sizes  = np.bincount(lbl.ravel()); sizes[0] = 0
    ridges = (sizes >= MIN_RIDGE_VOXELS)[lbl]; del lbl, sizes
    gwdt_arr = T_combined * ridges.astype(np.float32)   # soft proxy
    print(f'Fallback ridges: {ridges.sum():,} voxels  (T > {_thr:.4f})')

else:
    t0 = _time.time()

    # ── phi: signed field — negative = background seed ─────────────────────────
    # SEED_T_MAX ≈ 0: only exact-zero T voxels are seeds.
    # This is NOT a binarisation threshold — it merely identifies genuine background.
    phi = (T_combined.astype(np.float64) - SEED_T_MAX)

    n_seeds = int((phi < 0).sum())
    if n_seeds == 0:
        # T_combined has no exact zeros — use a tiny positive epsilon as threshold
        _fallback_seed = float(np.percentile(T_combined, 1))
        print(f'No T=0 voxels found. Using 1st percentile as seed threshold: {_fallback_seed:.5f}')
        phi = T_combined.astype(np.float64) - _fallback_seed
        n_seeds = int((phi < 0).sum())

    print(f'Background seeds (phi < 0) : {n_seeds:,}  ({100*n_seeds/T_combined.size:.2f}% of volume)')
    print(f'Foreground voxels (phi ≥ 0): {(phi >= 0).sum():,}')

    # ── Speed: bright → fast → low travel time cost ────────────────────────────
    # Tube centers: high T → high speed → short travel time from edge to center
    # Background:   T ≈ 0 → speed ≈ SPEED_EPS → very slow (high resistance)
    speed = (T_combined.astype(np.float64).clip(min=0) + SPEED_EPS) ** SPEED_ALPHA

    print(f'\nSpeed range: [{speed.min():.5f}, {speed.max():.5f}]')
    print(f'Running FMM on volume {T_combined.shape} ...')
    print(f'Estimated RAM: ~{T_combined.size * 8 / 1e9:.1f} GB (float64)')

    # ── FMM travel time → GWDT ─────────────────────────────────────────────────
    gwdt_ma = skfmm.travel_time(phi, speed=speed, dx=float(voxel_iso), order=1)
    del phi, speed
    gc.collect()

    # Background voxels (phi < 0) are masked in the result — fill with 0
    if hasattr(gwdt_ma, 'filled'):
        gwdt_arr = gwdt_ma.filled(0).astype(np.float32)
    else:
        gwdt_arr = np.array(gwdt_ma, dtype=np.float32)
        gwdt_arr[gwdt_arr < 0] = 0
    del gwdt_ma
    gc.collect()

    elapsed = _time.time() - t0
    print(f'\nFMM done in {elapsed:.1f}s  ({elapsed/60:.1f} min)')
    fg_vals = gwdt_arr[gwdt_arr > 0]
    print(f'GWDT range (foreground): [{fg_vals.min():.4f}, {fg_vals.max():.4f}]')
    print(f'Foreground voxels      : {len(fg_vals):,}')
    print()
    print('High GWDT values = voxel is deep inside a bright tube (tube centre).')
    print('Low  GWDT values = voxel is near the background edge (tube boundary).')

---
## Step 5b · Ridge Extraction — Local Maxima of GWDT

The medial axis of each tube corresponds to **local maxima** of the GWDT map:
these are voxels that are deeper inside the tube than all their 26 neighbours.

At a **crossing**, two tubes intersect. Because each fiber has its own T profile,
their GWDT ridges diverge at the crossing region — yielding two separate ridge
lines rather than the merged blob that binary skeletonisation produces.

In [ ]:
from scipy.ndimage import label as _ndlabel

# ── Local maximum in 26-neighbourhood ────────────────────────────────────────
# A voxel is a ridge point if gwdt_arr[v] >= gwdt_arr[neighbour] for all 26 neighbours.
# Using maximum_filter: ridges[v] = True iff gwdt_arr[v] == max in its 3³ window.
footprint = np.ones((3, 3, 3), dtype=bool)   # 26-connectivity footprint
gwdt_max  = maximum_filter(gwdt_arr, footprint=footprint, mode='constant', cval=0)
ridges    = (gwdt_arr > 0) & (gwdt_arr >= gwdt_max * (1.0 - 1e-5))
del gwdt_max

n_ridge_raw = int(ridges.sum())
print(f'Ridge voxels (local maxima of GWDT): {n_ridge_raw:,}')
print(f'  ({100*n_ridge_raw/T_combined.size:.3f}% of volume)')

# ── Size filter on ridge components ──────────────────────────────────────────
print(f'\nSize filter: removing components < {MIN_RIDGE_VOXELS} voxels ...')
lbl, n_comp = _ndlabel(ridges, structure=np.ones((3,3,3),dtype=int))
sizes       = np.bincount(lbl.ravel()); sizes[0] = 0
n_kept      = int((sizes >= MIN_RIDGE_VOXELS).sum())
n_removed   = int(n_comp - n_kept)
ridges      = (sizes >= MIN_RIDGE_VOXELS)[lbl]
del lbl, sizes

n_ridge_filt = int(ridges.sum())
print(f'  Components: {n_comp:,} total  →  {n_kept:,} kept  ({n_removed:,} removed)')
print(f'  Ridge voxels after filter: {n_ridge_filt:,}')

---
## Step 5c & 5d · Topology-Preserving Thinning

The ridge map may still be a few voxels thick at bifurcations or crossing regions.
Lee 1994 parallel thinning reduces it to a 1-voxel-thick centerline graph while
preserving topology (no branches broken, no loops added).

Because the ridge map is already thin (unlike the original binary candidate mask),
skeletonisation is fast and produces fewer spurious spurs.

In [ ]:
import time as _time2

print(f'Skeletonising {ridges.sum():,} ridge voxels ...')
t0 = _time2.time()
skeleton    = morphology.skeletonize(ridges)
skel_coords = np.argwhere(skeleton)
elapsed     = _time2.time() - t0
compression = 100 * (1 - skeleton.sum() / max(ridges.sum(), 1))

print(f'Done in {elapsed:.1f}s')
print(f'Ridge voxels  : {ridges.sum():,}')
print(f'Skeleton voxels: {skeleton.sum():,}  ({compression:.1f}% reduction)')
print(f'skel_coords   : {skel_coords.shape}')
print(f'Physical length: {skeleton.sum() * voxel_iso:.1f} µm')

In [ ]:
# MIP comparison: GWDT map  vs  skeleton
show_mip(gwdt_arr, title='GWDT map (arrival time — bright = tube centres)', cmap='hot')
compare_mip(ridges.astype(np.float32), skeleton.astype(np.float32),
            title_a='Ridge map (local maxima of GWDT)',
            title_b='Skeleton (1-voxel centerlines)')

In [ ]:
show_slices(stack_iso,
            title='Skeleton overlay — centerlines should follow fiber axes',
            overlay=skeleton.astype(np.float32),
            overlay_cmap='spring',
            overlay_thresh=0.5)

In [ ]:
import os
os.makedirs('output', exist_ok=True)

# ── Save checkpoint ───────────────────────────────────────────────────────────
# orient_field is passed through from the tubularity checkpoint so step3 can
# use it for direction-consistency in crossing separation without re-loading
# the full tubularity checkpoint.
np.savez_compressed(
    'output/skeleton.npz',
    gwdt_arr    = gwdt_arr,                # float32  (Z, Y, X) — GWDT arrival time map
    ridges      = ridges,                  # bool     (Z, Y, X) — local maxima of GWDT
    skeleton    = skeleton,                # bool     (Z, Y, X) — 1-voxel centerlines
    skel_coords = skel_coords,             # int64    (N, 3)    — (z,y,x) skeleton voxels
    radius_map  = radius_map,              # float32  (Z, Y, X) — local tube radius µm
    voxel_iso   = np.float32(voxel_iso),   # float32  scalar
    **({'orient_field': orient_field} if orient_field is not None else {}),
)

# TIFF outputs for inspection in Fiji/ImageJ
tifffile.imwrite('output/gwdt.tif',
                 (gwdt_arr / max(float(gwdt_arr.max()), 1e-10)).astype(np.float32),
                 photometric='minisblack')
tifffile.imwrite('output/skeleton.tif',
                 skeleton.astype(np.uint8) * 255,
                 photometric='minisblack')

print(f'Saved → output/skeleton.npz')
print(f'         gwdt_arr    : {gwdt_arr.shape}  float32')
print(f'         ridges      : {ridges.shape}  bool')
print(f'         skeleton    : {skeleton.shape}  bool')
print(f'         skel_coords : {skel_coords.shape}  int64')
print(f'         radius_map  : {radius_map.shape}  float32')
if orient_field is not None:
    print(f'         orient_field: {orient_field.shape}  float16')
print(f'Saved → output/gwdt.tif')
print(f'Saved → output/skeleton.tif')

# ── Sanity checks ─────────────────────────────────────────────────────────────
assert skeleton.dtype == bool
assert skeleton.sum() <= ridges.sum()
assert skel_coords.shape[1] == 3
assert skel_coords.shape[0] == skeleton.sum()
assert gwdt_arr.min() >= 0
print('\nAll assertions passed.')

---
## Pipeline Status

| Step | Notebook | Status | Description |
|------|----------|--------|-------------|
| 1–4 | step1 | ✅ | Preprocess, multi-scale tubularity + orient_field |
| 5a | **step2** | ✅ | GWDT via FMM — no binarisation, seeds = T≈0 voxels |
| 5b | **step2** | ✅ | Ridge extraction (local maxima of GWDT) |
| 5c–d | **step2** | ✅ | Size filter + Lee thinning → 1-voxel skeleton |
| 6–11 | step3 | … | Soma detection, competitive FMM, SWC export |

### What step3 loads from `skeleton.npz`

```python
ck          = np.load('output/skeleton.npz')
skeleton    = ck['skeleton']      # bool (Z,Y,X)
skel_coords = ck['skel_coords']   # int64 (N, 3)
radius_map  = ck['radius_map']    # float32 (Z,Y,X)
voxel_iso   = float(ck['voxel_iso'])
gwdt_arr    = ck['gwdt_arr']      # float32 (Z,Y,X) — tube depth map
orient_field = ck.get('orient_field')  # float16 (Z,Y,X,3) or None
```

### GWDT tuning guide

| Symptom | Adjustment |
|---------|-----------|
| "No interface found" error from FMM | Raise `SEED_T_MAX` (e.g. 0 → 0.005) |
| Background noise appears in ridges | Raise `SPEED_EPS` (reduces contrast) → use `SPEED_ALPHA` > 1 instead |
| Ridges too thick at crossings | Raise `SPEED_ALPHA` (e.g. 1.0 → 1.5) |
| Short spurs on skeleton | Raise `MIN_RIDGE_VOXELS` (e.g. 50 → 100) |
| Thin fibers missing | Lower `SEED_T_MAX` (closer to 0) or lower `SPEED_EPS` |